# JEPX market making: calibration and time extension

A public-data study based on Kanamura and Bunn (2022), *Market making and electricity price formation in Japan*.

This notebook separates two questions: can the original sample's data logic be reconstructed, and do its patterns persist in a later market environment?

## Setup

Raw JEPX and weather files are public but are not stored in this repository. This cell locates the project locally or in Google Colab, then downloads them.

In [ ]:
from pathlib import Path
import subprocess
import sys

slug = 'kanamura-bunn-2022-jepx'
project_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / 'replications' / slug,
    Path.cwd() / 'paper-replications' / 'replications' / slug,
    Path('/content/paper-replications/replications') / slug,
]
project = next((path for path in project_candidates if (path / 'scripts' / 'download_data.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Could not locate the replication project directory.')

subprocess.run([sys.executable, str(project / 'scripts' / 'download_data.py')], check=True, cwd=project)

# Part A — Original-period calibration

The paper's displayed sample is FY2015–FY2019. This stage uses 2015-04-01 through 2020-03-08. It does not create new data; it checks that the variable construction and descriptive premise can be independently rebuilt.

Market tightness is `buy bid volume - sell bid volume`. As in the paper, Tokyo temperature is a first-order proxy for the national JEPX system price. This implementation uses the midpoint of daily maximum and minimum temperature, an explicit approximation.

In [ ]:
subprocess.run([sys.executable, str(project / 'scripts' / 'build_panel.py'), '--period', 'original'], check=True, cwd=project)
subprocess.run([sys.executable, str(project / 'scripts' / 'descriptive_analysis.py'), '--period', 'original'], check=True, cwd=project)

## Calibration result

| Fiscal year | Mean price (JPY/kWh) | Mean buy bids (million kWh) | Mean sell bids (million kWh) | Mean tightness (million kWh) |
| --- | ---: | ---: | ---: | ---: |
| FY2015 | 9.777 | 2.110 | 5.063 | -2.953 |
| FY2016 | 8.463 | 3.474 | 5.529 | -2.055 |
| FY2017 | 9.719 | 6.528 | 7.941 | -1.413 |
| FY2018 | 9.756 | 15.470 | 15.681 | -0.211 |
| FY2019 | 8.067 | 20.065 | 19.885 | 0.181 |

Buy and sell activity both grew strongly and became more balanced. This does not establish a causal effect or reproduce the paper's state-switching parameters.

![Original-period time series](../figures/original/descriptive_time_series.svg)

![Original-period temperature and tightness](../figures/original/temperature_scarcity.svg)

# Part B — Time-extension analysis

This is the research-facing section. The identical pipeline is applied to FY2020–FY2024, from 2020-04-01 through 2025-03-31. It includes the January 2021 price spike.

The question is not whether later parameters equal the original paper's parameters. It is whether the temperature, bid-imbalance, and price relationships appear stable or show evidence of a changed market regime.

In [ ]:
subprocess.run([sys.executable, str(project / 'scripts' / 'build_panel.py'), '--period', 'extension'], check=True, cwd=project)
subprocess.run([sys.executable, str(project / 'scripts' / 'descriptive_analysis.py'), '--period', 'extension'], check=True, cwd=project)

## Extension result

| Fiscal year | Mean price (JPY/kWh) | 95th percentile price | Mean buy bids (million kWh) | Mean sell bids (million kWh) | Mean tightness (million kWh) |
| --- | ---: | ---: | ---: | ---: | ---: |
| FY2020 | 11.209 | 42.000 | 21.444 | 21.831 | -0.387 |
| FY2021 | 13.457 | 30.000 | 22.879 | 22.855 | 0.024 |
| FY2022 | 20.406 | 39.720 | 21.051 | 23.209 | -2.158 |
| FY2023 | 10.742 | 17.126 | 18.370 | 22.788 | -4.418 |
| FY2024 | 12.293 | 20.000 | 19.529 | 24.509 | -4.980 |

The extension raises a nontrivial question: the bid-balance pattern that moved close to zero by FY2019 did not remain stable. FY2023 and FY2024 show substantially more negative average tightness, while price levels and upper-tail prices vary sharply. This is descriptive evidence, not yet a causal claim.

![Extension-period time series](../figures/extension/descriptive_time_series.svg)

![Extension-period temperature and tightness](../figures/extension/temperature_scarcity.svg)

# Next analytical step

Estimate the same piecewise V-shape relationship between temperature and market tightness in both periods, then compare the turning point and slopes. Only after that should we attempt the full state-switching model. The extension claim should be framed as a test of temporal stability, with market-design changes and extreme events treated as competing explanations.